# From a raw General Ledger to the full set of Financial Statements
### A step-by-step, "teach you how I solved it" walkthrough

**The mission.** We are handed one Excel workbook with a company's raw bookkeeping data for
**2018-2020**. The goal is to produce the four core financial statements that every accountant,
CFO and financial analyst relies on:

| Statement | Question it answers |
|---|---|
| **Income Statement (P&L)** | How much profit did the company make? |
| **Balance Sheet** | What does it own and owe at year-end? |
| **Statement of Changes in Equity (SoCE)** | How did owners' equity move during the year? |
| **Cash Flow Statement** | Where did actual cash come from and go? |

**The twist:** the workbook does NOT contain the statements. It only contains the *ingredients*:
a general ledger (GL) with ~28,000 bookkeeping lines, a chart of accounts, and two statement
*templates* that tell us which line of each statement every account belongs to. Our job is to
assemble the statements ourselves and **prove they are internally consistent** (every statement
must reconcile with every other statement).

**How I'll teach it:** I'll take you through exactly the detective work I did:
inspect the raw data -> decode how it is structured -> build each statement -> and finish with
a reconciliation "scorecard" where every cross-check must come out at exactly zero. Along the
way I'll show the *mistakes I made and how I caught them*, because that is where the real
learning happens.

## Step 1 - Inventory the data: what is in the workbook?

Before writing any logic, I always open every sheet and ask *what is this table for?*

- **`GL`** - the General Ledger: one row per bookkeeping line (28k rows). Every economic event
  (a sale, a purchase, a salary payment, ...) is recorded here with a date, a territory, an
  account number (`Account_key`) and a signed amount. This is the *only* table with numbers.
- **`Chart of Accounts`** - the dictionary: it maps each `Account_key` to a name and tells us
  *where it belongs* in the statements (Balance Sheet asset vs liability vs equity, or which
  P&L bucket).
- **`CashFlow_St`** and **`SoCE_St`** - the two report *templates*: they define the layout of
  the Cash Flow statement and the Statement of Changes in Equity, and which accounts feed each
  line.
- **`Territory`** and **`Calendar`** - dimension tables (country/region per territory; date
  attributes per day). Useful later for analysis by region.

Let's load everything and take a first look.

In [1]:
import pandas as pd
import numpy as np
from pathlib import Path

FILE = Path('Finacial_Data.xlsx')      # run this notebook from the Finance folder

xl = pd.ExcelFile(FILE)
gl         = pd.read_excel(xl, 'GL')
coa        = pd.read_excel(xl, 'Chart of Accounts')
cf_tpl     = pd.read_excel(xl, 'CashFlow_St')
soce_tpl   = pd.read_excel(xl, 'SoCE_St')
territory  = pd.read_excel(xl, 'Territory')
calendar   = pd.read_excel(xl, 'Calendar')

print('Sheets:', xl.sheet_names)
for name, df in [('GL', gl), ('Chart of Accounts', coa), ('CashFlow_St', cf_tpl),
                 ('SoCE_St', soce_tpl), ('Territory', territory), ('Calendar', calendar)]:
    print(f'  {name:<18s} {df.shape[0]:>6d} rows x {df.shape[1]:>2d} columns')

Sheets: ['CashFlow_St', 'SoCE_St', 'GL', 'Chart of Accounts', 'Territory', 'Calendar']
  GL                  27910 rows x  8 columns
  Chart of Accounts      54 rows x  7 columns
  CashFlow_St            66 rows x 11 columns
  SoCE_St                13 rows x  6 columns
  Territory               7 rows x  3 columns
  Calendar             1096 rows x  5 columns


In [2]:
gl.head(6)

,Index,Date,Territory_key,Account_key,Details,Amount,Unnamed: 6,Unnamed: 7
0,1.1,2018-01-01,1.0,230.0,Cost of Sales,-884,NaN,NaN
1,1.2,2018-01-01,1.0,60.0,Cost of Sales,-884,NaN,NaN
2,2.1,2018-01-01,1.0,230.0,Cost of Sales,-1120,NaN,NaN
3,2.2,2018-01-01,1.0,60.0,Cost of Sales,-1120,NaN,NaN
4,3.1,2018-01-01,1.0,280.0,Credit Expenses,-2394,NaN,NaN
5,3.2,2018-01-01,1.0,120.0,Credit Expenses,2394,NaN,NaN


In [3]:
coa.head(12)

,Account_key,Report,Class,SubClass,SubClass2,Account,SubAccount
0,10,Balance Sheet,Assets,Assets,Current Assets,Cash & Cash Equivalents,Cash at Bank
1,20,Balance Sheet,Assets,Assets,Current Assets,Cash & Cash Equivalents,Cash in hand
2,30,Balance Sheet,Assets,Assets,Current Assets,Receivables,Trade Receivables
3,40,Balance Sheet,Assets,Assets,Current Assets,Receivables,Other Receivables
4,41,Balance Sheet,Assets,Assets,Current Assets,Receivables,Interest Receivable
5,42,Balance Sheet,Assets,Assets,Current Assets,Receivables,Dividend Receivable
6,50,Balance Sheet,Assets,Assets,Current Assets,Receivables,Allowance for Bad Debt
7,60,Balance Sheet,Assets,Assets,Current Assets,Inventory,Inventory
8,70,Balance Sheet,Assets,Assets,Current Assets,Other Current Assets,Prepaid Expenses
9,75,Balance Sheet,Assets,Assets,Current Assets,Investments,Securities


## Step 2 - Clean the GL: the footer row and the Index column

Two things stand out immediately:

1. **There is a footer row** at the bottom of `GL` with no `Index`, no date and a giant total
   in `Amount` (it is the grand total of the whole file). Keeping it would corrupt every sum,
   so we drop rows without an `Index`.
2. **The `Index` column encodes structure.** Values like `1.1`, `1.2`, `2.1` mean
   *transaction number . line number*: `1.1` and `1.2` are the two lines of transaction 1,
   `2.1`/`2.2` the two lines of transaction 2, and so on. Splitting this column lets us reason
   about whole transactions, which matters for the next step.

Let me also extract a `Year` column from `Date` - the statements are yearly.

In [4]:
# 1) drop the footer row: it has no Index and holds a grand total
print('footer row ->', gl[gl['Index'].isna()].to_dict('records'))
gl_clean = gl.dropna(subset=['Index']).copy()

# 2) parse '1.1' into transaction number and line number
gl_clean['txn']   = gl_clean['Index'].astype(str).str.split('.').str[0].astype(int)
gl_clean['line']  = gl_clean['Index'].astype(str).str.split('.').str[1].astype(int)
gl_clean['Year']  = pd.to_datetime(gl_clean['Date']).dt.year
gl_clean['Account_key'] = gl_clean['Account_key'].astype(int)

print('rows kept:', len(gl_clean))
print('date range:', gl_clean['Date'].min().date(), '->', gl_clean['Date'].max().date())
print('years:', sorted(gl_clean['Year'].unique()))
print('lines per transaction:', gl_clean.groupby('txn')['line'].max().value_counts().to_dict())

footer row -> [{'Index': nan, 'Date': NaT, 'Territory_key': nan, 'Account_key': nan, 'Details': nan, 'Amount': 24640002, 'Unnamed: 6': nan, 'Unnamed: 7': 5461.0}]
rows kept: 27909
date range: 2018-01-01 -> 2020-12-31
years: [np.int32(2018), np.int32(2019), np.int32(2020)]
lines per transaction: {2: 1984, 3: 8}


In [5]:
# A whole transaction, side by side with the account dictionary
sample = gl_clean[(gl_clean['txn'].isin([1, 3, 4, 5])) & (gl_clean['Territory_key'] == 1)]
sample.merge(coa[['Account_key', 'Report', 'Class', 'Account']], on='Account_key', how='left')[
    ['Index', 'Details', 'Account_key', 'Account', 'Report', 'Class', 'Amount']]

,Index,Details,Account_key,Account,Report,Class,Amount
0,1.1,Cost of Sales,230,Cost of Sales,Profit and Loss,Trading account,-884
1,1.2,Cost of Sales,60,Inventory,Balance Sheet,Assets,-884
2,3.1,Credit Expenses,280,Advertisements,Profit and Loss,Operating account,-2394
3,3.2,Credit Expenses,120,Other Payables,Balance Sheet,Liabilities and Owners Equity,2394
4,4.1,Credit Sales,210,Sales,Profit and Loss,Trading account,2948
5,4.2,Credit Sales,30,Receivables,Balance Sheet,Assets,2948
6,5.1,Cash Sales,210,Sales,Profit and Loss,Trading account,3734
7,5.2,Cash Sales,10,Cash & Cash Equivalents,Balance Sheet,Assets,3734


## Step 3 - The detective work: decoding the sign convention

This is the single most important step of the whole exercise, so I'll go slowly.

Look at the four transactions above. In a *normal* accounting ledger, every transaction has
equal debits and credits, so its lines sum to **zero**. Here they do not:

| Txn | Lines | Accounting logic |
|---|---|---|
| `3` Credit Expenses | expense `-2394`, accrued liability `+2394` | sum 0 - looks normal |
| `4` Credit Sales | sales `+2948`, receivables `+2948` | sum `+5896` - *not* zero! |
| `5` Cash Sales | sales `+3734`, cash `+3734` | sum `+7468` - *not* zero! |
| `1` Cost of Sales | cost of sales `-884`, inventory `-884` | sum `-1768` - *not* zero! |

So this ledger is **not** stored as debit/credit. What convention is it using then?

**My hypothesis:** each amount is stored in the *natural statement sign* - the sign the figure
would carry in the financial statements themselves:

* **Assets:** positive = the asset *increased*, negative = it decreased
* **Liabilities / Equity:** positive = increased, negative = decreased
* **Revenue / income:** positive = earned, negative = reversed
* **Expenses / tax / depreciation:** negative = incurred (they reduce profit)

Test it against the table: cash sales *increase* cash (`+`) and *earn* sales (`+`) - matches.
Credit expenses *incur* an expense (`-`) and *increase* a liability (`+`) - matches.
Cost of sales *increases* the expense (`-`) and *decreases* inventory (`-`) - matches.

The beauty of this convention is a **conservation law**: every transaction must preserve

$$\text{Assets} = \text{Liabilities} + \text{Equity} + \text{Revenue} + \text{Expenses}$$

where Revenue/Expenses are signed exactly as in the data. If my hypothesis is right, the left
and right sides of this equation must be equal for *every single transaction* in the file
(plus one extra bucket: account 1010 "Adjusting", which the company uses to close the P&L
into Retained Earnings at year-end). Let's test it on a sample of transactions.

In [6]:
# classify every account: Balance-Sheet side or P&L side (revenue vs expense)
bs_side, pnl_side = {}, {}
for k, rep, cls in coa[['Account_key', 'Report', 'Class']].itertuples(index=False):
    k = int(k)
    if rep == 'Balance Sheet':
        bs_side[k] = 'Asset' if cls == 'Assets' else ('Equity' if cls == 'Owners Equity' else 'Liability')
    elif rep == 'Profit and Loss':
        pnl_side[k] = 'Revenue' if k in (210, 410, 420, 430, 431) else 'Expense'

def txn_check(txn_id, territory_key):
    """Test the conservation law  A = L + E + Rev + Exp + Adj  for one transaction.

    'Adj' is account 1010 (the year-end P&L clearing account) - it only appears in
    the profit-transfer transactions and completes the identity there."""
    rows = gl_clean[(gl_clean['txn'] == txn_id) & (gl_clean['Territory_key'] == territory_key)]
    a = l = e = rev = exp = adj = 0.0
    for _, r in rows.iterrows():
        k = int(r['Account_key'])
        if k in bs_side:
            a   += r['Amount'] if bs_side[k] == 'Asset'     else 0
            l   += r['Amount'] if bs_side[k] == 'Liability' else 0
            e   += r['Amount'] if bs_side[k] == 'Equity'    else 0
        elif k in pnl_side:
            rev += r['Amount'] if pnl_side[k] == 'Revenue' else 0
            exp += r['Amount'] if pnl_side[k] == 'Expense' else 0
        else:                      # account 1010 - Adjusting
            adj += r['Amount']
    lhs, rhs = a, l + e + rev + exp + adj
    ok = abs(lhs - rhs) < 0.005
    desc = rows['Details'].iloc[0]
    legs = '; '.join(f"acct {int(r['Account_key'])}: {r['Amount']:,.0f}" for _, r in rows.iterrows())
    print(f"txn {txn_id:>4} [{desc:<28s}] {legs:<45s} A={lhs:>10,.0f}  L+E+R+Ex+Adj={rhs:>10,.0f}  -> {'PASS' if ok else 'FAIL'}")

for t, terr in [(1, 1), (3, 1), (4, 1), (5, 1), (8, 1), (12, 1), (26, 1), (50, 1), (963, 1), (49, 1)]:
    txn_check(t, terr)

txn    1 [Cost of Sales               ] acct 230: -884; acct 60: -884                 A=      -884  L+E+R+Ex+Adj=      -884  -> PASS
txn    3 [Credit Expenses             ] acct 280: -2,394; acct 120: 2,394             A=         0  L+E+R+Ex+Adj=         0  -> PASS
txn    4 [Credit Sales                ] acct 210: 2,948; acct 30: 2,948               A=     2,948  L+E+R+Ex+Adj=     2,948  -> PASS
txn    5 [Cash Sales                  ] acct 210: 3,734; acct 10: 3,734               A=     3,734  L+E+R+Ex+Adj=     3,734  -> PASS
txn    8 [Bank to Cash transfer       ] acct 20: 15,000; acct 10: -15,000             A=         0  L+E+R+Ex+Adj=         0  -> PASS
txn   12 [Share Issue                 ] acct 10: 800,000; acct 180: 800,000           A=   800,000  L+E+R+Ex+Adj=   800,000  -> PASS
txn   26 [Purchase of shares          ] acct 75: 80,000; acct 10: -80,000             A=         0  L+E+R+Ex+Adj=         0  -> PASS
txn   50 [Depreciation for the month  ] acct 380: -10,000; acct 90: -

Every transaction passes. The natural-sign convention is confirmed, and we now have a
**mathematical guarantee**: because every transaction preserves
`A = L + E + Revenue + Expenses`, any statement we build by summing these signed amounts must
stay consistent. (It also explains why the grand total of the file is not zero - the file is
not a debit/credit ledger.)

Two more useful facts discovered along the way:

* The `x.1` line of each transaction is *not* a "header to discard" - both lines are real
  postings (the transaction above with `txn=1` posts to both account 230 and account 60).
* There are 7 territories; every transaction is tagged with one of them, so we can also
  produce *per-territory* statements later.

## Step 4 - Build the account x year movements matrix

Everything we need flows from one aggregation: **the total movement of each account in each
year** (sum of `Amount` per `Account_key` and `Year`). Joined with the chart of accounts, this
single table is the raw material for every statement.

> Note: 2018 starts from scratch - there are no opening-balance entries in the GL, so the
> 2018 movement *is* the 2018 closing balance; later years accumulate on top.

In [7]:
mov = (gl_clean
       .groupby(['Account_key', 'Year'])['Amount']
       .sum()
       .unstack(fill_value=0.0))
mov.columns.name = None

acct_info = coa[['Account_key', 'Report', 'Class', 'SubClass', 'Account']].set_index('Account_key')
mov = acct_info.join(mov).sort_index()
mov[[2018, 2019, 2020]] = mov[[2018, 2019, 2020]].fillna(0.0)   # accounts with no GL activity = 0

YEARS = [2018, 2019, 2020]
mov.head(10).map(lambda v: f'{v:,.0f}' if isinstance(v, (int, float)) else v)

,Report,Class,SubClass,Account,2018,2019,2020
Account_key,,,,,,,
10,Balance Sheet,Assets,Assets,Cash & Cash Equivalents,"2,056,505","2,839,487","-1,086,116"
20,Balance Sheet,Assets,Assets,Cash & Cash Equivalents,"51,186","163,836","485,834"
30,Balance Sheet,Assets,Assets,Receivables,"551,614","228,848","473,532"
40,Balance Sheet,Assets,Assets,Receivables,0,0,0
41,Balance Sheet,Assets,Assets,Receivables,0,0,0
42,Balance Sheet,Assets,Assets,Receivables,0,0,0
50,Balance Sheet,Assets,Assets,Receivables,0,0,0
60,Balance Sheet,Assets,Assets,Inventory,"105,834","416,529","803,879"
70,Balance Sheet,Assets,Assets,Other Current Assets,0,0,0


Now the year-level sanity check. Because every transaction obeys the conservation law,
the *totals* must too - including the `Adjusting` account 1010, which the company uses at
year-end to transfer the year's profit into Retained Earnings (its balance mirrors the P&L):

$$\text{Assets} = \text{Liabilities} + \text{Equity} + \text{P\&L} + \text{Adjusting}$$

In [8]:
def year_identity_check(y):
    m = mov[['Report', 'Class', y]]
    assets  = m.loc[m['Class'] == 'Assets', y].sum()
    liab    = m.loc[m['Class'] == 'Liabilities', y].sum()
    equity  = m.loc[m['Class'] == 'Owners Equity', y].sum()
    pnl     = m.loc[m['Report'] == 'Profit and Loss', y].sum()
    adj     = m.loc[m['Report'] == 'Adjusting', y].sum()
    return {'Year': y, 'Assets': assets, 'Liab+Equity+P&L+Adj': liab + equity + pnl + adj,
            'Gap (must be 0)': assets - (liab + equity + pnl + adj)}

checks = pd.DataFrame([year_identity_check(y) for y in YEARS]).set_index('Year')
checks.map(lambda v: f'{v:,.0f}')

,Assets,Liab+Equity+P&L+Adj,Gap (must be 0)
Year,,,
2018,"3,875,802",0,"3,875,802"
2019,"5,362,294",0,"5,362,294"
2020,"3,081,905",0,"3,081,905"


## Step 5 - The Income Statement (P&L)

**The logic.** In the natural-sign world, the P&L is beautifully simple: *revenue accounts are
positive, cost accounts are negative, and the sum is profit.* We just need to arrange the
accounts in the standard layout using the chart of accounts:

```
Sales (210) + Sales returns (220)                 -> Net sales
- Cost of sales (230)                             -> Gross profit
- Operating expenses (240..360)                   -> EBITDA
- Depreciation (370..390) - Amortisation (400)    -> Operating profit (EBIT)
+ Non-operating income (410, 420, 430, 431) - Interest (440)
                                                  -> Profit before tax
- Taxation (450)                                  -> NET PROFIT
```

Because expenses are stored negative, the "subtractions" are just additions of negative
numbers - one sum, no sign juggling.

In [9]:
def pnl_year(y):
    m = mov[y]
    s = lambda k: m.get(k, 0.0)
    d = {}
    d['Sales']                          = s(210)
    d['Sales returns']                  = s(220)
    d['Net sales']                      = d['Sales'] + d['Sales returns']
    d['Cost of sales']                  = s(230)
    d['Gross profit']                   = d['Net sales'] + d['Cost of sales']
    d['Operating expenses (detail)']    = sum(s(k) for k in range(240, 361, 10))
    d['EBITDA']                         = d['Gross profit'] + d['Operating expenses (detail)']
    d['Depreciation']                   = s(370) + s(380) + s(390)
    d['Amortisation']                   = s(400)
    d['Operating profit (EBIT)']        = d['EBITDA'] + d['Depreciation'] + d['Amortisation']
    d['Interest income']                = s(410)
    d['Gain on sale of assets']         = s(420)
    d['Exchange gain/(loss)']           = s(430)
    d['Dividend income']                = s(431)
    d['Interest expense']               = s(440)
    d['Profit before tax']              = (d['Operating profit (EBIT)'] + d['Interest income']
                                           + d['Gain on sale of assets'] + d['Exchange gain/(loss)']
                                           + d['Dividend income'] + d['Interest expense'])
    d['Taxation']                       = s(450)
    d['Net profit']                     = d['Profit before tax'] + d['Taxation']
    return d

pnl = pd.DataFrame({y: pnl_year(y) for y in YEARS})
pnl.map(lambda v: f'{v:,.0f}')

,2018,2019,2020
Sales,"3,694,374","5,856,210","8,005,347"
Sales returns,"-118,946","-158,365","-169,978"
Net sales,"3,575,428","5,697,845","7,835,369"
Cost of sales,"-1,192,182","-1,729,299","-2,494,009"
Gross profit,"2,383,246","3,968,546","5,341,360"
Operating expenses (detail),"-1,235,441","-1,960,802","-3,104,354"
EBITDA,"1,147,805","2,007,744","2,237,006"
Depreciation,"-387,996","-516,600","-698,880"
Amortisation,"-19,008","-15,456","-15,960"
Operating profit (EBIT),"740,801","1,475,688","1,522,166"


In [10]:
# ...and the detail behind 'Operating expenses'
opex_keys = [(k, coa.set_index('Account_key').loc[k, 'Account']) for k in range(240, 361, 10)]
opex = pd.DataFrame({label: {y: mov[y].get(k, 0.0) for y in YEARS} for k, label in opex_keys}).T
opex['Total'] = opex.sum(axis=1)
opex.map(lambda v: f'{v:,.0f}')

,2018,2019,2020,Total
Staff Costs,"-260,004","-374,412","-527,436","-1,161,852"
Bad Debt Expense,0,0,0,0
Commissions,"-290,549","-497,368","-824,697","-1,612,614"
Conferences,0,0,0,0
Advertisements,"-52,539","-83,555","-134,431","-270,525"
Travel,"-94,728","-151,157","-242,865","-488,750"
Entertainment,"-135,167","-214,247","-345,168","-694,582"
Office Supplies,"-63,921","-101,655","-163,552","-329,128"
Professional Services,"-88,439","-141,053","-226,663","-456,155"
Telephone,"-76,018","-120,765","-194,380","-391,163"


In [11]:
# KEY RECONCILIATION #1: Net profit must equal the amount transferred
# into Retained Earnings (account 200) each year.
for y in YEARS:
    delta_re = mov.loc[200, y]
    np_ = pnl[y]['Net profit']
    print(f'{y}:  P&L net profit {np_:>12,.0f}   vs   movement in Retained Earnings {delta_re:>12,.0f}   ->',
          'MATCH' if abs(np_ - delta_re) < 0.5 else 'MISMATCH')

2018:  P&L net profit      623,856   vs   movement in Retained Earnings      623,856   -> MATCH
2019:  P&L net profit    1,303,147   vs   movement in Retained Earnings    1,303,147   -> MATCH
2020:  P&L net profit    1,289,945   vs   movement in Retained Earnings    1,289,945   -> MATCH


## Step 6 - The Balance Sheet

**The logic.** The Balance Sheet is a *stock*, not a flow: each year's figure is the
**cumulative** total of all movements up to and including that year (2018 starts from zero).
The natural signs do all the work: assets are positive when they grow, liabilities and equity
positive when they grow, so the classic equation simply reads

$$\text{Total assets} = \text{Total liabilities} + \text{Total equity}$$

In [12]:
def cumulative(accts, y):
    """Sum of movements of the given accounts up to and including year y."""
    return gl_clean[(gl_clean['Account_key'].isin(accts)) & (gl_clean['Year'] <= y)]['Amount'].sum()

def bs_year(y):
    c = lambda k: cumulative([k], y)
    d = {}
    d['Cash at bank']                     = c(10)
    d['Cash in hand']                     = c(20)
    d['Trade receivables']                = c(30)
    d['Other receivables']                = c(40)
    d['Inventory']                        = c(60)
    d['Prepaid expenses']                 = c(70)
    d['Securities (investments)']         = c(75)
    d['Property, plant & equipment']      = c(80) + c(90)
    d['Intangible assets']                = c(100)
    d['TOTAL ASSETS']                     = (d['Cash at bank'] + d['Cash in hand'] + d['Trade receivables']
                                            + d['Other receivables'] + d['Inventory'] + d['Prepaid expenses']
                                            + d['Securities (investments)'] + d['Property, plant & equipment']
                                            + d['Intangible assets'])
    d['Trade payables']                   = c(110)
    d['Accrued expenses']                 = c(120)
    d['Interest payable']                 = c(121)
    d['Tax payable']                      = c(122)
    d['Salaries payable']                 = c(130)
    d['Dividends payable']                = c(135)
    d['Other current liabilities']        = c(140) + c(150)
    d['Long-term loans']                  = c(160) + c(170)
    d['TOTAL LIABILITIES']                = (d['Trade payables'] + d['Accrued expenses'] + d['Interest payable']
                                            + d['Tax payable'] + d['Salaries payable'] + d['Dividends payable']
                                            + d['Other current liabilities'] + d['Long-term loans'])
    d['Share capital']                    = c(180)
    d['Share premium']                    = c(190)
    d['Retained earnings']                = c(200)
    d['Dividends paid (contra)']          = c(201)
    d['TOTAL EQUITY']                     = (d['Share capital'] + d['Share premium']
                                            + d['Retained earnings'] + d['Dividends paid (contra)'])
    d['TOTAL LIABILITIES + EQUITY']       = d['TOTAL LIABILITIES'] + d['TOTAL EQUITY']
    return d

bs = pd.DataFrame({y: bs_year(y) for y in YEARS})
bs.map(lambda v: f'{v:,.0f}')

,2018,2019,2020
Cash at bank,"2,056,505","4,895,992","3,809,876"
Cash in hand,"51,186","215,022","700,856"
Trade receivables,"551,614","780,462","1,253,994"
Other receivables,0,0,0
Inventory,"105,834","522,363","1,326,242"
Prepaid expenses,0,0,0
Securities (investments),"258,667","642,667","1,314,667"
"Property, plant & equipment","776,004","2,028,804","3,677,220"
Intangible assets,"75,992","152,786","237,146"
TOTAL ASSETS,"3,875,802","9,238,096","12,320,001"


In [13]:
# KEY RECONCILIATION #2: the balance sheet must balance every year.
for y in YEARS:
    a = bs[y]['TOTAL ASSETS']
    le = bs[y]['TOTAL LIABILITIES + EQUITY']
    print(f'{y}:  Assets {a:>12,.0f}   vs   Liabilities + Equity {le:>12,.0f}   ->',
          'BALANCED' if abs(a - le) < 0.5 else 'OUT OF BALANCE')

2018:  Assets    3,875,802   vs   Liabilities + Equity    3,875,802   -> BALANCED
2019:  Assets    9,238,096   vs   Liabilities + Equity    9,238,096   -> BALANCED
2020:  Assets   12,320,001   vs   Liabilities + Equity   12,320,001   -> BALANCED


## Step 7 - Statement of Changes in Equity (SoCE)

**The logic.** The workbook ships a template (`SoCE_St`) that tells us exactly which rows and
accounts make up this statement. The story it tells for each year is:

```
Balance at the beginning          (prior year's closing balances)
+ Issue of share capital          (movement in 180 + 190)
+ Total income for the year       (movement in 200  =  the P&L net profit)
- Dividends                       (movement in 201  =  dividends announced)
= Balance at the end              (which must equal the Balance Sheet equity)
```

(The template also has a 'Changes in accounting policy' row, but the GL contains no such
entries, so it stays zero.)

In [14]:
def soce_year(y):
    prev = y - 1
    opening = lambda k: cumulative([k], prev)      # 0 for 2018, by design
    closing = lambda k: cumulative([k], y)
    m = mov[y]
    g = lambda k: m.get(k, 0.0)
    return {
        'Balance at the beginning': [opening(180), opening(190), opening(200), opening(201)],
        'Issue of share capital':   [g(180),      g(190),      0,            0],
        'Total income for the year':[0,           0,            g(200),       0],
        'Dividends':                [0,           0,            0,            g(201)],
        'Balance at the end':       [closing(180), closing(190), closing(200), closing(201)],
    }

cols = ['Share capital', 'Share premium', 'Retained earnings', 'Dividends paid', 'Total']
rows_order = ['Balance at the beginning', 'Issue of share capital',
              'Total income for the year', 'Dividends', 'Balance at the end']

data = {y: [] for y in YEARS}
for label in rows_order:
    for y in YEARS:
        vals = soce_year(y)[label]
        data[y].extend(vals + [sum(vals)])

soce = pd.DataFrame(data, index=pd.MultiIndex.from_product([rows_order, cols]))
soce.map(lambda v: f'{v:,.0f}')

2018       2019        2020
Balance at the beginning  Share capital              0  2,666,667   6,462,667
                          Share premium              0          0           0
                          Retained earnings          0    623,856   1,927,003
                          Dividends paid             0   -259,667    -579,467
                          Total                      0  3,030,856   7,810,203
Issue of share capital    Share capital      2,666,667  3,796,000   1,643,200
                          Share premium              0          0           0
                          Retained earnings          0          0           0
                          Dividends paid             0          0           0
                          Total              2,666,667  3,796,000   1,643,200
Total income for the year Share capital              0          0           0
                          Share premium              0          0           0
                          Retained earnings    623,856  1,303,147   1,289,945
                          Dividends paid             0          0           0
                          Total                623,856  1,303,147   1,289,945
Dividends                 Share capital              0          0           0
                          Share premium              0          0           0
                          Retained earnings          0          0           0
                          Dividends paid      -259,667   -319,800    -394,992
                          Total               -259,667   -319,800    -394,992
Balance at the end        Share capital      2,666,667  6,462,667   8,105,867
                          Share premium              0          0           0
                          Retained earnings    623,856  1,927,003   3,216,948
                          Dividends paid      -259,667   -579,467    -974,459
                          Total              3,030,856  7,810,203  10,348,356

In [15]:
# KEY RECONCILIATION #3: SoCE closing total must equal Balance-Sheet equity.
for y in YEARS:
    soce_close = soce[y].loc[('Balance at the end', 'Total')]
    bs_equity  = bs[y]['TOTAL EQUITY']
    print(f'{y}:  SoCE closing equity {soce_close:>12,.0f}   vs   BS equity {bs_equity:>12,.0f}   ->',
          'MATCH' if abs(soce_close - bs_equity) < 0.5 else 'MISMATCH')

2018:  SoCE closing equity    3,030,856   vs   BS equity    3,030,856   -> MATCH
2019:  SoCE closing equity    7,810,203   vs   BS equity    7,810,203   -> MATCH
2020:  SoCE closing equity   10,348,356   vs   BS equity   10,348,356   -> MATCH


## Step 8 - Cash Flow Statement, method 1: the DIRECT method

**The logic.** "Cash flow" only happens when one of the two *cash* accounts (10 = cash at
bank, 20 = cash in hand) is touched. So the direct method is brutally simple:

1. Take every GL row that hits a cash account.
2. Classify each one by its `Details` description into Operating / Investing / Financing.
3. Sum.

- **Operating** = the day-to-day business: cash sales, collections from customers, payments
  to suppliers, salaries, cash expenses, tax payments, ...
- **Investing** = buying/selling long-term assets and receiving interest/dividends.
- **Financing** = raising capital (shares, loans) and paying dividends.
- `Bank to Cash transfer` is internal (bank -> till), so it nets to zero and is excluded.

This method must reproduce the movement of cash *exactly* - it is a pure re-arrangement of the
cash rows, nothing more.

In [16]:
OP_DETAILS = ['Cash Sales', 'Cash Received from Debtors', 'Cash expenses', 'Salaries',
              'Payment - Credit Expenses', 'Sales Return', 'Tax payment for the previous year',
              'Interest expense', 'Exchange gain/loss', 'Bad Debts', 'Inventory Purchase']
INV_DETAILS = ['Purchase of equipment', 'Purchase of shares', 'Purchase of intangibles assets',
               'Sale of asset', 'Dividend income', 'Interest income']
FIN_DETAILS = ['Share Issue', 'New loan raised @ 6%', 'Payment of final dividends',
               'Payment of interim dividends']

# cash movement of every Detail type, per year
cash_by_detail = (gl_clean[gl_clean['Account_key'].isin([10, 20])]
                  .groupby(['Year', 'Details'])['Amount'].sum())

def direct_cf(y):
    c = cash_by_detail.xs(y)
    d = {}
    for detail in OP_DETAILS:
        d[detail] = c.get(detail, 0.0)
    d['NET CASH FROM OPERATING'] = sum(d[k] for k in OP_DETAILS)
    for detail in INV_DETAILS:
        d[detail] = c.get(detail, 0.0)
    d['NET CASH FROM INVESTING'] = sum(d[k] for k in INV_DETAILS)
    for detail in FIN_DETAILS:
        d[detail] = c.get(detail, 0.0)
    d['NET CASH FROM FINANCING'] = sum(d[k] for k in FIN_DETAILS)
    d['NET CHANGE IN CASH'] = (d['NET CASH FROM OPERATING'] + d['NET CASH FROM INVESTING']
                               + d['NET CASH FROM FINANCING'])
    d['CASH AT START OF YEAR'] = cumulative([10, 20], y - 1)
    d['CASH AT END OF YEAR'] = d['CASH AT START OF YEAR'] + d['NET CHANGE IN CASH']
    return d

cf_direct = pd.DataFrame({y: direct_cf(y) for y in YEARS})
cf_direct.map(lambda v: f'{v:,.0f}')

,2018,2019,2020
Cash Sales,"673,342","1,096,000","1,546,573"
Cash Received from Debtors,"2,469,418","4,531,362","5,985,242"
Cash expenses,"-10,803","-22,982","-33,144"
Salaries,"-260,004","-374,412","-527,436"
Payment - Credit Expenses,"-865,527","-1,523,875","-2,633,892"
Sales Return,"-118,946","-158,365","-169,978"
Tax payment for the previous year,0,"-137,572","-198,108"
Interest expense,"-14,520","-21,000","-24,979"
Exchange gain/loss,"2,214","3,705","5,016"
Bad Debts,"47,602","39,183","39,356"


In [17]:
# KEY RECONCILIATION #4: the direct cash flow must equal the cash in the Balance Sheet.
for y in YEARS:
    cf_cash = cf_direct[y]['CASH AT END OF YEAR']
    bs_cash = bs[y]['Cash at bank'] + bs[y]['Cash in hand']
    print(f'{y}:  CF closing cash {cf_cash:>12,.0f}   vs   BS cash {bs_cash:>12,.0f}   ->',
          'MATCH' if abs(cf_cash - bs_cash) < 0.5 else 'MISMATCH')

2018:  CF closing cash    2,107,691   vs   BS cash    2,107,691   -> MATCH
2019:  CF closing cash    5,111,014   vs   BS cash    5,111,014   -> MATCH
2020:  CF closing cash    4,510,732   vs   BS cash    4,510,732   -> MATCH


## Step 9 - Cash Flow Statement, method 2: the INDIRECT method

**Why a second method?** The direct method is easy but it doesn't *explain* the cash flow:
it just re-sorts cash rows. The indirect method (the one used in real annual reports, and the
one the workbook's template describes) starts from **profit** and shows *why* profit differs
from cash. That is a far more insightful question.

**The recipe, step by step:**

1. **Start from Profit before tax** (PBT = net profit + tax).
2. **Add back non-cash expenses** - depreciation and amortisation reduce profit but never
   touch cash, so add them back.
3. **Remove non-operating income** - interest income, dividend income and gains on asset
   sales are not operating flows; we will report them down in Investing instead.
4. **Working capital changes** - profit is earned on an accruals basis: selling on credit
   raises profit but not cash (receivables up -> cash down), buying inventory uses cash
   (inventory up -> cash down), delaying payments preserves cash (payables up -> cash up).
   Rule of thumb: *an increase in an asset consumes cash; an increase in a liability
   provides cash*.
5. **Interest and tax paid** - re-express the expense as actual cash paid by adjusting for
   the change in the payable account.
6. **Investing and financing** as before, but derived from balance-sheet movements
   (careful: intangible purchases were financed by a loan, so they never touched cash!).

*Where I went wrong while building this (and how I caught it):* I first computed the asset
purchases from the net movement of the PP&E account - but that movement also includes
**depreciation** (a non-cash reduction) and loan-financed intangibles. The reconciliation
check failed, which forced me to trace every line back to its `Details` type. Lesson: always
let a hard reconciliation drive you to the correct logic - never trust a formula that
"looks right". 

In [18]:
def bs_delta(accts, y):
    """Change in the cumulative balance of accounts between year y-1 and y."""
    return cumulative(accts, y) - cumulative(accts, y - 1)

def indirect_cf(y):
    m = mov[y]
    s = lambda k: m.get(k, 0.0)
    pbt      = pnl[y]['Profit before tax']

    # 1-3) non-cash and non-operating adjustments (flip the natural sign: expenses are negative)
    depr     = -(s(370) + s(380) + s(390))
    amort    = -(s(400))
    int_inc  = s(410)
    div_inc  = s(431)
    gain     = s(420)

    # 4) working capital changes
    wc_receivables = -bs_delta([30, 40, 41, 42], y)
    wc_inventory   = -bs_delta([60], y)
    wc_prepaid     = -bs_delta([70], y)
    wc_payables    =  bs_delta([110, 120, 130, 140], y)

    # 5) interest and tax actually paid
    int_exp      = s(440)
    interest_paid = s(440) + bs_delta([121], y)      # expense + change in payable
    tax_paid      = s(450) + bs_delta([122], y)

    operating = (pbt + depr + amort - int_inc - div_inc - gain - int_exp
                 + wc_receivables + wc_inventory + wc_prepaid + wc_payables
                 + interest_paid + tax_paid)

    # 6) investing & financing, derived from the *cash legs* of the transactions
    yr = gl_clean[gl_clean['Year'] == y]
    buy_equipment  = -yr[(yr['Details'] == 'Purchase of equipment') & (yr['Account_key'] == 90)]['Amount'].sum()
    buy_shares     = -yr[(yr['Details'] == 'Purchase of shares') & (yr['Account_key'] == 75)]['Amount'].sum()
    buy_intang     = -yr[(yr['Details'] == 'Purchase of intangibles assets') & (yr['Account_key'] == 10)]['Amount'].sum()  # 0: financed by loan
    sale_proceeds  = -yr[(yr['Details'] == 'Sale of asset') & (yr['Account_key'] == 90)]['Amount'].sum() + gain
    investing      = buy_equipment + buy_shares + buy_intang + sale_proceeds + div_inc + int_inc

    share_issues   = bs_delta([180, 190], y)
    loan_proceeds  = yr[(yr['Details'] == 'New loan raised @ 6%') & (yr['Account_key'] == 160)]['Amount'].sum()
    dividends_paid = bs_delta([135, 201], y)
    financing      = share_issues + loan_proceeds + dividends_paid

    return {
        'Profit before tax':          pbt,
        '+ Depreciation':             depr,
        '+ Amortisation':             amort,
        '- Interest income':          -int_inc,
        '- Dividend income':          -div_inc,
        '- Gain on sale of assets':   -gain,
        'Working capital: receivables': wc_receivables,
        'Working capital: inventory':   wc_inventory,
        'Working capital: prepaids':    wc_prepaid,
        'Working capital: payables':    wc_payables,
        'Interest paid':              interest_paid,
        'Tax paid':                   tax_paid,
        'NET CASH FROM OPERATING':    operating,
        'Purchase of equipment':      buy_equipment,
        'Purchase of shares':         buy_shares,
        'Purchase of intangibles':    buy_intang,
        'Proceeds from sale of assets': sale_proceeds,
        'Dividends received':         div_inc,
        'Interest received':          int_inc,
        'NET CASH FROM INVESTING':    investing,
        'Proceeds from share issues': share_issues,
        'Proceeds from loans':        loan_proceeds,
        'Dividends paid':             dividends_paid,
        'NET CASH FROM FINANCING':    financing,
        'NET CHANGE IN CASH':         operating + investing + financing,
    }

cf_indirect = pd.DataFrame({y: indirect_cf(y) for y in YEARS})
cf_indirect.map(lambda v: f'{v:,.0f}')

,2018,2019,2020
Profit before tax,"761,438","1,501,250","1,569,027"
+ Depreciation,"387,996","516,600","698,880"
+ Amortisation,"19,008","15,456","15,960"
- Interest income,"-13,496","-16,621","-30,868"
- Dividend income,"-15,917","-21,751","-30,030"
- Gain on sale of assets,"-4,850","-5,085","-6,351"
Working capital: receivables,"-551,614","-228,848","-473,532"
Working capital: inventory,"-105,834","-416,529","-803,879"
Working capital: prepaids,0,0,0
Working capital: payables,"284,377","179,866","80,404"


In [19]:
# KEY RECONCILIATION #5: the two methods must agree with each other AND with the BS.
for y in YEARS:
    i = cf_indirect[y]['NET CHANGE IN CASH']
    d = cf_direct[y]['NET CHANGE IN CASH']
    b = bs[y]['Cash at bank'] + bs[y]['Cash in hand'] - (cumulative([10, 20], y - 1))
    print(f'{y}:  indirect {i:>12,.0f}   direct {d:>12,.0f}   BS change {b:>12,.0f}   ->',
          'ALL MATCH' if (abs(i - d) < 0.5 and abs(i - b) < 0.5) else 'MISMATCH')

2018:  indirect    2,107,691   direct    2,107,691   BS change    2,107,691   -> ALL MATCH
2019:  indirect    3,003,323   direct    3,003,323   BS change    3,003,323   -> ALL MATCH
2020:  indirect     -600,282   direct     -600,282   BS change     -600,282   -> ALL MATCH


## Step 10 - The reconciliation scorecard

Real financial modelling is only as good as its cross-checks. Every statement we built was
derived from the *same* GL, so they must agree with each other in five independent ways.
If any of these ever fails, there is a bug somewhere - this is how I caught my own mistakes
while building this notebook.

In [20]:
scorecard = []
for y in YEARS:
    bs_gap    = bs[y]['TOTAL ASSETS'] - bs[y]['TOTAL LIABILITIES + EQUITY']
    re_gap    = pnl[y]['Net profit'] - mov.loc[200, y]
    cf_gap    = cf_direct[y]['NET CHANGE IN CASH'] - cf_indirect[y]['NET CHANGE IN CASH']
    cfbs_gap  = cf_direct[y]['NET CHANGE IN CASH'] - (bs[y]['Cash at bank'] + bs[y]['Cash in hand']
                                                      - cumulative([10, 20], y - 1))
    soce_gap  = soce[y].loc[('Balance at the end', 'Total')] - bs[y]['TOTAL EQUITY']
    identity  = year_identity_check(y)['Gap (must be 0)']
    scorecard.append({
        'Year': y,
        'BS balances (A - L - E)':              bs_gap,
        'P&L -> Retained Earnings':             re_gap,
        'CF direct vs indirect':                cf_gap,
        'CF vs BS cash movement':               cfbs_gap,
        'SoCE closing vs BS equity':            soce_gap,
        'Ledger identity (A-L-E-P&L-Adj)':      identity,
    })
score = pd.DataFrame(scorecard).set_index('Year')
score.map(lambda v: f'{v:,.0f}')

,BS balances (A - L - E),P&L -> Retained Earnings,CF direct vs indirect,CF vs BS cash movement,SoCE closing vs BS equity,Ledger identity (A-L-E-P&L-Adj)
Year,,,,,,
2018,0,0,0,0,0,"3,875,802"
2019,0,0,0,0,0,"5,362,294"
2020,0,0,0,0,0,"3,081,905"


**Every check is exactly zero.** The statements are mathematically consistent with each
other and with the raw ledger. That zero row is the professional's "sign-off".

## Step 11 - Bonus: slice by territory and region

Because the GL tags every transaction with a `Territory_key`, all of the machinery above can
be re-run per territory. Let's do a compact version: the 2020 P&L and a balance-sheet check
for each territory, then roll up by region.

In [21]:
# 2020 income statement per territory (Sales, Net profit) + region roll-up
terr_pnl = []
for t, trow in territory.iterrows():
    g = gl_clean[(gl_clean['Year'] == 2020) & (gl_clean['Territory_key'] == trow['Territory_key'])]
    by_acc = g.groupby('Account_key')['Amount'].sum()
    sales = by_acc.get(210, 0.0)
    pnl_keys = set(coa[coa['Report'] == 'Profit and Loss']['Account_key'])
    net = by_acc[by_acc.index.isin(pnl_keys)].sum()
    terr_pnl.append({'Territory': trow['Country'], 'Region': trow['Region'],
                     'Sales 2020': sales, 'Net profit 2020': net})
terr = pd.DataFrame(terr_pnl).set_index('Territory')
terr.loc['TOTAL GROUP'] = terr.sum(numeric_only=True)
terr['Region'] = terr['Region'].fillna('')
terr.map(lambda v: f'{v:,.0f}' if isinstance(v, (int, float)) else v)

,Region,Sales 2020,Net profit 2020
Territory,,,
USA,North America,"2,466,897","437,934"
Canada,North America,"493,387","87,593"
UK,Europe,"435,407","-42,951"
Germany,Europe,"1,154,556","166,496"
France,Europe,"1,006,542","261,600"
Australia,Oceania,"1,006,542","-189,659"
New Zealand,Oceania,"1,442,016","568,932"
TOTAL GROUP,,"8,005,347","1,289,945"


In [22]:
# Does every territory's balance sheet balance on its own? (cumulative to 2020)
print(f"{'Territory':<10} {'Assets':>12} {'Liab+Equity':>12} {'Gap':>10}")
for t, trow in territory.iterrows():
    g = gl_clean[gl_clean['Territory_key'] == trow['Territory_key']]
    by_acc = g.groupby('Account_key')['Amount'].sum()
    assets  = sum(v for k, v in by_acc.items() if k in bs_side and bs_side[k] == 'Asset')
    liab    = sum(v for k, v in by_acc.items() if k in bs_side and bs_side[k] == 'Liability')
    equity  = sum(v for k, v in by_acc.items() if k in bs_side and bs_side[k] == 'Equity')
    print(f'{trow["Country"]:<10} {assets:>12,.0f} {liab + equity:>12,.0f} {(assets - liab - equity):>10,.0f}')

Territory        Assets  Liab+Equity        Gap
USA           3,716,693    3,716,693          0
Canada          965,622      965,622          0
UK            1,075,493    1,075,493          0
Germany       1,700,660    1,700,660          0
France        1,582,355    1,582,355          0
Australia     1,150,351    1,150,351          0
New Zealand    2,128,827    2,128,827          0


## Summary - what we did and why it works

1. **Inspected the data** - six sheets, but only the GL holds numbers; the chart of accounts
   and the two templates are the *reporting dictionary*.
2. **Cleaned and decoded the GL** - dropped the grand-total footer, split `Index` into
   transaction/line, and - the crucial step - worked out that amounts are stored in
   **natural statement signs** (assets/revenue positive when they increase; expenses and
   reductions negative). I *proved* this convention by testing the conservation law
   `Assets = Liabilities + Equity + Revenue + Expenses` on individual transactions.
3. **One aggregation to rule them all** - account x year movements, which feeds every
   statement.
4. **Built the four statements**:
   * **P&L** - natural signs make it a pure sum of mapped accounts; net profit = the
     Retained-Earnings transfer.
   * **Balance Sheet** - cumulative movements; balances every year.
   * **SoCE** - implemented straight from the workbook template; closes onto the BS equity.
   * **Cash Flow** - built it *twice*: the direct method (re-sorting cash rows) and the
     indirect method (profit -> cash via non-cash, non-operating, working-capital,
     interest and tax adjustments). Both agree, and both agree with the BS cash.
5. **Reconciliation scorecard** - six independent cross-checks, all exactly zero.

**What you can try next (great practice):**

* Re-run the statements **per territory** (I only did a compact 2020 version above).
* Build the **quarterly** P&L using the `Calendar` sheet (Qtr column).
* Reproduce the workbook's `CashFlow_St` template literally and reconcile it against the
  direct method - some of its `ValueType` rules ('Positive'/'Negative' variants) need the
  same careful treatment we applied to the indirect method.
* Write the statements out to a multi-sheet Excel file with `pd.ExcelWriter`.

*The single most valuable habit in this whole exercise:* never ship a financial statement
without the scorecard - let reconciliations drive your logic, and treat any non-zero gap as a
bug until proven otherwise.